# 03 · Gold · `kpis_mensal`
### V-Energy Grid 360 — Arquitetura Medalhão

 Depende da `gold.fato_interrupcao` 

Responde ao pedido do diretor: *"ver os principais indicadores do negócio e compará-los entre períodos, para saber rapidamente se estamos indo bem ou mal"*. Grava `v_energy.gold.kpis_mensal`, seguindo o **contrato da Gold (tabela A5, versão v4)**, com as 20 colunas do contrato.

**Grão:** 1 linha = 1 mês de referência (`2022-07` a `2026-06`, 48 linhas). É o resumo da **empresa inteira**; os recortes por município, bairro e classe ficam com a `fato_segmento_mensal` (A3).

**Fontes (Gold):** `fato_consumo_faturamento` (consumo, receita, bandeira, inadimplência, GD), `dim_uc` (UCs cadastradas), `fato_atendimento` (chamados e nota) e `fato_interrupcao` (interrupções, DEC e FEC). **Tudo vem das tabelas fato da Gold, e não das Silvers**, para que o painel e a visão 360 mostrem o mesmo número.

| Coluna | Regra |
|---|---|
| `qtd_ucs_cadastradas` | UCs de `dim_uc` com `data_ligacao` até o último dia do mês |
| `qtd_ucs_faturadas` | UCs distintas com fatura no mês |
| `consumo_total_kwh`, `receita_energia`, `receita_total` | somas de `consumo_kwh`, `valor_energia` e `valor_total` |
| `preco_medio_kwh` | `receita_energia / consumo_total_kwh` |
| `bandeira_predominante`, `pct_faturas_bandeira_nao_verde` | bandeira com mais faturas no mês; % das faturas com bandeira diferente de `Verde` |
| `qtd_faturas_em_aberto`, `inadimplencia_pct` | faturas **desta competência** em aberto; % do `valor_total` **desta competência** em aberto |
| `valor_em_aberto_mais_90_dias` | `valor_total` desta competência em aberto na faixa `MAIS DE 90 DIAS` |
| `qtd_ucs_com_gd` | UCs com `possui_gd_no_mes = true` no mês (a regra "GD no mês" das outras tabelas) |
| `qtd_interrupcoes`, `dec_horas`, `fec` | ocorrências do mês; `minutos / 60 / qtd_ucs_cadastradas`; `ocorrências / qtd_ucs_cadastradas` |
| `qtd_atendimentos`, `nota_satisfacao_media` | chamados **com abertura** no mês; média das notas **preenchidas** |
| `variacao_consumo_ano_anterior_pct`, `variacao_receita_ano_anterior_pct` | contra o mesmo mês do ano anterior (receita = `receita_energia`); **nulas** de `2022-07` a `2023-06` |

**O que entra no dicionário de dados desta tabela:**
- **A inadimplência é por competência**, nunca um número único misturando anos: ~8% das faturas de cada mês ficam em aberto, em todos os anos. **Não use `dias_atraso` das faturas pagas** como inadimplência (83% das pagas têm algum atraso);
- **`valor_em_aberto_mais_90_dias` é zero nos últimos meses** por construção: o atraso das faturas em aberto é contado até `2026-06-30`, e as de maio e junho de 2026 ainda não passaram de 90 dias;
- **`qtd_atendimentos` conta os chamados com abertura no mês.** `2025-07` tem **18.061**, cerca de 2,8× o normal (evento climático; log A6). Não há chamados em aberto nesta base;
- **`dec_horas` e `fec` usam como denominador a `qtd_ucs_cadastradas` do próprio mês**, e não a do alimentador (essa é a `dec_fec_alimentador_mensal`);
- **`bandeira_predominante`**: parece que todas as faturas de um mês têm a mesma bandeira nesta base (os 66,67% de faturas não verdes dos últimos 12 meses, que aparecem em vários segmentos da `receita_decomposicao`, sugerem 8 meses não verdes em 12). **Não foi verificado em todos os meses**: a tabela da seção 7 mostra `pct_faturas_bandeira_nao_verde` mês a mês, e valores diferentes de 0 e 100 desmentem a hipótese;
- **`nota_satisfacao_media`** não inclui os chamados sem avaliação (3,0%).

## 1. Parâmetros do ambiente

In [0]:
catalog = "v_energy"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"
landing_path = f"/Volumes/{catalog}/bronze/landing"

from decimal import Decimal
from pyspark.sql import functions as F, Row, Window
from datetime import datetime

tabela_fato = f"{gold_schema}.fato_consumo_faturamento"
tabela_dim = f"{gold_schema}.dim_uc"
tabela_atend = f"{gold_schema}.fato_atendimento"
tabela_intr = f"{gold_schema}.fato_interrupcao"
tabela_stg = f"{gold_schema}.kpis_mensal_stg"
tabela_gold = f"{gold_schema}.kpis_mensal"
tabela_dq_log = f"{silver_schema}.dq_log"      # mesmo log das Silvers e das outras Golds

P0_INI, P0_FIM = "2022-07", "2023-06"          # primeiro ano
P1_INI, P1_FIM = "2025-07", "2026-06"          # últimos 12 meses

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
print(f"origem:  {tabela_fato}, {tabela_dim}, {tabela_atend}, {tabela_intr}")
print(f"destino: {tabela_gold} (via {tabela_stg})")

origem:  v_energy.gold.fato_consumo_faturamento, v_energy.gold.dim_uc, v_energy.gold.fato_atendimento, v_energy.gold.fato_interrupcao
destino: v_energy.gold.kpis_mensal (via v_energy.gold.kpis_mensal_stg)


## 2. Funções de checagem de qualidade

Mesmas funções da aula e das Silvers (`dq_check`, `dq_check_unique` e `dq_check_contagem`), com as mesmas colunas no log (`table_name`, `check_name`, `total_rows`, `failed_rows`, `passed`, `checked_at`). Acrescento `dq_check_valor`, para comparar um número calculado com um esperado (somas que precisam bater com as Silvers e com o diagnóstico); ela grava no log nas mesmas colunas, com `failed_rows` igual a 0 ou 1.

As funções só acumulam os resultados em `dq_results`. Depois das checagens, o notebook **grava o resultado em `silver.dq_log`** (sempre) e **só então decide se grava a Gold**.

**Cuidado com nulos:** linhas em que a condição resulta em nulo não contam como falha. Quando o nulo também for um problema, há um `dq_check` separado com `isNotNull()`.

In [0]:
dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=failed, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=dupes, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

def n_br(n):
    return f"{n:,}".replace(",", ".")

def dq_check_contagem(table_name: str, check_name: str, df, esperado: int):
    """Checagem de contagem total: passa só se o número de linhas for exatamente o esperado."""
    total = df.count()
    diferenca = abs(total - esperado)
    passed = diferenca == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=diferenca, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {n_br(total)} linhas (esperado {n_br(esperado)})")

def dq_check_valor(table_name: str, check_name: str, total_rows: int, obtido, esperado):
    """Compara um valor calculado com o esperado (soma que precisa bater com a Silver ou com o diagnóstico)."""
    passed = obtido == esperado
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total_rows,
            failed_rows=0 if passed else 1, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | obtido {obtido} (esperado {esperado})")

## 3. Leitura das fontes e conferência das colunas

Antes de construir, o notebook confere que as quatro tabelas existem **com as colunas de que precisa**. 
Contagens esperadas: **2.152.336** linhas na fato de consumo, **45.000** UCs, **318.871** chamados e **180.000** interrupções (uma por UC interrompida).

In [0]:
exigidas = {
    tabela_fato: ["id_uc", "mes_referencia", "consumo_kwh", "valor_energia", "valor_total", "bandeira_tarifaria",
                  "em_aberto", "faixa_atraso", "possui_gd_no_mes"],
    tabela_dim: ["id_uc", "data_ligacao"],
    tabela_atend: ["mes_referencia", "nota_satisfacao"],
    tabela_intr: ["mes_referencia", "duracao_min"],
}
for tabela, colunas in exigidas.items():
    try:
        existentes = spark.table(tabela).columns
    except Exception as e:
        raise Exception(f"{tabela} não encontrada. Rode antes o notebook que a gera (a fato_interrupcao é da Aimê).") from e
    faltando = [c for c in colunas if c not in existentes]
    if faltando:
        raise Exception(f"{tabela} não tem as colunas {faltando} (DESCRIBE {tabela}).")

fato = spark.table(tabela_fato)
dim = spark.table(tabela_dim)
atend = spark.table(tabela_atend)
intr = spark.table(tabela_intr)
n_fato, n_dim, n_atend, n_intr = fato.count(), dim.count(), atend.count(), intr.count()
print(f"fato: {n_br(n_fato)} | UCs: {n_br(n_dim)} | chamados: {n_br(n_atend)} | interrupções: {n_br(n_intr)}")
# esperado: 2.152.336 | 45.000 | 318.871 | 180.000

fato: 2.152.336 | UCs: 45.000 | chamados: 318.871 | interrupções: 180.000


## 4. Os indicadores de cada mês

**Decisões:**
- **O mês vem da fato de consumo** (os 48 meses de `2022-07` a `2026-06`). Os indicadores de atendimento e de interrupção entram por `left join`, e mês sem ocorrência vale **zero**, e não nulo.
- **Inadimplência** = `valor_total` das faturas em aberto **da própria competência** ÷ `valor_total` da competência. A dívida de meses anteriores não entra.
- **`qtd_ucs_cadastradas`**: junção por data (`data_ligacao` até o último dia do mês). A base cresce, então o valor aumenta com o tempo.
- **Bandeira predominante**: a com mais faturas, desempatando pelo nome.
- **Variação contra o ano anterior** por junção com o **mesmo mês do ano anterior** (nula quando o mês não existe, ou seja, nos 12 primeiros).

**Cuidados:** as divisões usam `try_divide` (o modo ANSI do serverless interrompe a divisão por zero), e o consumo e a receita vêm de `decimal` para as somas ficarem exatas.

In [0]:
def dec(coluna, p, s):
    return F.col(coluna).cast(f"decimal({p},{s})")

# --- indicadores de faturamento, por mês
fat_mes = (fato.groupBy("mes_referencia").agg(
    F.countDistinct("id_uc").alias("qtd_ucs_faturadas"),
    F.sum("consumo_kwh").alias("consumo_total_kwh"),
    F.sum("valor_energia").alias("receita_energia"),
    F.sum("valor_total").alias("receita_total"),
    F.sum(F.when(F.col("bandeira_tarifaria") != "Verde", 1).otherwise(0)).alias("faturas_nao_verde"),
    F.count("*").alias("faturas"),
    F.sum(F.col("em_aberto").cast("int")).alias("qtd_faturas_em_aberto"),
    F.sum(F.when(F.col("em_aberto"), F.col("valor_total")).otherwise(0)).alias("valor_em_aberto"),
    F.sum(F.when(F.col("em_aberto") & (F.col("faixa_atraso") == "MAIS DE 90 DIAS"), F.col("valor_total")).otherwise(0)).alias("valor_em_aberto_mais_90_dias"),
    F.sum(F.col("possui_gd_no_mes").cast("int")).alias("qtd_ucs_com_gd"),
))

# --- bandeira predominante
w_band = Window.partitionBy("mes_referencia").orderBy(F.desc("n"), F.asc("bandeira_tarifaria"))
band = (fato.groupBy("mes_referencia", "bandeira_tarifaria").agg(F.count("*").alias("n"))
        .withColumn("rn", F.row_number().over(w_band)).filter("rn = 1")
        .select("mes_referencia", F.col("bandeira_tarifaria").alias("bandeira_predominante")))

# --- UCs cadastradas até o fim de cada mês
meses = (fat_mes.select("mes_referencia")
         .withColumn("fim_mes", F.last_day(F.to_date(F.concat("mes_referencia", F.lit("-01"))))))
cad = (meses.join(dim.select("id_uc", "data_ligacao"), F.col("data_ligacao") <= F.col("fim_mes"), "left")
       .groupBy("mes_referencia").agg(F.count("id_uc").alias("qtd_ucs_cadastradas")))

# --- interrupções e atendimentos, por mês de abertura
intr_mes = intr.groupBy("mes_referencia").agg(F.count("*").alias("qtd_interrupcoes"), F.sum("duracao_min").alias("minutos_interrompidos"))
atend_mes = atend.groupBy("mes_referencia").agg(F.count("*").alias("qtd_atendimentos"),
                                                F.avg(F.col("nota_satisfacao").cast("double")).alias("nota_satisfacao_media"))

base = (fat_mes.join(band, "mes_referencia", "left").join(cad, "mes_referencia", "left")
        .join(intr_mes, "mes_referencia", "left").join(atend_mes, "mes_referencia", "left")
        .withColumn("qtd_interrupcoes", F.coalesce("qtd_interrupcoes", F.lit(0)))
        .withColumn("minutos_interrompidos", F.coalesce("minutos_interrompidos", F.lit(0)))
        .withColumn("qtd_atendimentos", F.coalesce("qtd_atendimentos", F.lit(0))))

# --- mesmo mês do ano anterior (para as duas variações)
ano_ant = base.select(
    F.concat((F.substring("mes_referencia", 1, 4).cast("int") + 1).cast("string"), F.lit("-"), F.substring("mes_referencia", 6, 2)).alias("mes_referencia"),
    F.col("consumo_total_kwh").alias("consumo_ant"), F.col("receita_energia").alias("receita_ant"))

kpis = (base.join(ano_ant, "mes_referencia", "left")
        .withColumn("preco_medio_kwh", F.round(F.expr("try_divide(receita_energia, consumo_total_kwh)"), 4))
        .withColumn("pct_faturas_bandeira_nao_verde", F.round(F.expr("try_divide(faturas_nao_verde, faturas)") * 100, 2))
        .withColumn("inadimplencia_pct", F.round(F.expr("try_divide(valor_em_aberto, receita_total)") * 100, 2))
        .withColumn("dec_horas", F.round(F.expr("try_divide(minutos_interrompidos / 60.0, qtd_ucs_cadastradas)"), 4))
        .withColumn("fec", F.round(F.expr("try_divide(qtd_interrupcoes, qtd_ucs_cadastradas)"), 4))
        .withColumn("variacao_consumo_ano_anterior_pct", F.round(F.expr("try_divide(consumo_total_kwh - consumo_ant, consumo_ant)") * 100, 2))
        .withColumn("variacao_receita_ano_anterior_pct", F.round(F.expr("try_divide(receita_energia - receita_ant, receita_ant)") * 100, 2)))

df_kpis = kpis.select(
    "mes_referencia", "qtd_ucs_cadastradas", "qtd_ucs_faturadas",
    dec("consumo_total_kwh", 18, 2).alias("consumo_total_kwh"),
    dec("receita_energia", 18, 2).alias("receita_energia"),
    dec("receita_total", 18, 2).alias("receita_total"),
    dec("preco_medio_kwh", 12, 4).alias("preco_medio_kwh"),
    "bandeira_predominante",
    dec("pct_faturas_bandeira_nao_verde", 5, 2).alias("pct_faturas_bandeira_nao_verde"),
    "qtd_faturas_em_aberto",
    dec("inadimplencia_pct", 5, 2).alias("inadimplencia_pct"),
    dec("valor_em_aberto_mais_90_dias", 18, 2).alias("valor_em_aberto_mais_90_dias"),
    "qtd_ucs_com_gd", "qtd_interrupcoes",
    dec("dec_horas", 10, 4).alias("dec_horas"),
    dec("fec", 10, 4).alias("fec"),
    "qtd_atendimentos",
    dec("nota_satisfacao_media", 4, 2).alias("nota_satisfacao_media"),
    dec("variacao_consumo_ano_anterior_pct", 8, 2).alias("variacao_consumo_ano_anterior_pct"),
    dec("variacao_receita_ano_anterior_pct", 8, 2).alias("variacao_receita_ano_anterior_pct"),
)
print("colunas:", len(df_kpis.columns))   # esperado: 20

# Grava na tabela temporária (48 linhas); as checagens leem dela
(
    df_kpis.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_stg)
)
k = spark.table(tabela_stg)
print(f"{tabela_stg}: {k.count()} linhas")      # esperado: 48
display(k.orderBy("mes_referencia"))

colunas: 20
v_energy.gold.kpis_mensal_stg: 48 linhas


mes_referencia,qtd_ucs_cadastradas,qtd_ucs_faturadas,consumo_total_kwh,receita_energia,receita_total,preco_medio_kwh,bandeira_predominante,pct_faturas_bandeira_nao_verde,qtd_faturas_em_aberto,inadimplencia_pct,valor_em_aberto_mais_90_dias,qtd_ucs_com_gd,qtd_interrupcoes,dec_horas,fec,qtd_atendimentos,nota_satisfacao_media,variacao_consumo_ano_anterior_pct,variacao_receita_ano_anterior_pct
2022-07,44644,44638,14099929.00,7142109.38,9522812.38,0.5065,Verde,0.00,3582,5.35,509464.28,0,3333,0.1170,0.0747,6322,4.10,null,null
2022-08,44653,44644,14166862.00,7175795.81,9567726.59,0.5065,Verde,0.00,3665,5.15,492512.64,0,3341,0.1267,0.0748,6474,4.10,null,null
2022-09,44669,44653,15560451.00,7914910.64,10553214.57,0.5087,Verde,0.00,3471,5.23,551533.82,0,3415,0.1244,0.0765,6309,4.10,null,null
2022-10,44677,44669,18433287.00,9440337.45,12587116.57,0.5121,Verde,0.00,3563,5.69,715635.45,0,3274,0.1197,0.0733,6326,4.10,null,null
2022-11,44685,44677,20854327.00,10733548.79,14311398.20,0.5147,Verde,0.00,3592,5.43,777749.98,0,3433,0.1265,0.0768,6467,4.10,null,null
2022-12,44693,44685,21833062.00,11252215.41,15002953.69,0.5154,Verde,0.00,3599,5.47,820674.15,0,3372,0.1255,0.0754,6377,4.10,null,null
2023-01,44704,44693,22714349.00,11721284.60,15628379.70,0.5160,Verde,0.00,3605,5.68,888126.89,0,3336,0.1251,0.0746,6323,4.11,null,null
2023-02,44711,44704,22022368.00,11352937.95,15137250.60,0.5155,Verde,0.00,3598,5.86,887367.40,0,3318,0.1250,0.0742,6448,4.10,null,null
2023-03,44720,44711,22233809.00,11464886.79,15286516.05,0.5157,Verde,0.00,3548,5.74,877944.04,0,3293,0.1215,0.0736,6315,4.10,null,null
2023-04,44732,44720,20915567.00,10765712.15,14354282.22,0.5147,Verde,0.00,3545,5.97,856671.54,0,3343,0.1236,0.0747,6482,4.10,null,null


## 5. Checagens de qualidade

Rodam sobre a tabela temporária, **antes** de gravar a final:

| Checagem | Protege |
|---|---|
| 48 meses, de `2022-07` a `2026-06`, sem repetir nem pular | o grão |
| Colunas não nulas (as variações só podem ser nulas nos 12 primeiros meses) | junções |
| `inadimplencia_pct` e `pct_faturas_bandeira_nao_verde` entre 0 e 100; nota entre 1 e 5 | faixas |
| `qtd_ucs_cadastradas` e `qtd_ucs_com_gd` **nunca diminuem** de um mês para o seguinte | coerência |
| **Soma do consumo e da receita dos 48 meses = a da fato** | nada se perdeu |
| **Volume e receita dos dois períodos = os do diagnóstico** (227.052.927 e 218.079.054 kWh; R$ 117.830.394,38 e R$ 127.262.595,65) | reproduz os números do diretor |
| **Faturas em aberto, em todos os meses = 172.553** | inadimplência |
| **Chamados: 318.871 no total e 18.061 em `2025-07`** | atendimento |
| **Interrupções: 180.000 no total e 17.844 em `2025-07`** (o mês do evento climático) | continuidade |
| **UCs com GD em `2026-06` = 3.510** e UCs cadastradas no último mês = 45.000 | GD e cadastro |
| `bandeira_predominante` = `Verde` e 0% não verdes no primeiro ano | bandeira |
| `dec_horas` e `fec` = 0 nos meses sem interrupção | continuidade |

Os valores esperados valem para esta extração; numa reexecução com dados novos, precisam ser atualizados.

In [0]:
t = "gold.kpis_mensal"
dq_results.clear()
n_k = k.count()

# Grão: 48 meses contínuos
dq_check_valor(t, "48 meses", n_k, n_k, 48)
dq_check_unique(t, "mes_referencia único", k, ["mes_referencia"])
lim = k.agg(F.min("mes_referencia"), F.max("mes_referencia")).first()
dq_check_valor(t, "primeiro mês = 2022-07", n_k, lim[0], "2022-07")
dq_check_valor(t, "último mês = 2026-06", n_k, lim[1], "2026-06")
dq_check(t, "mes_referencia no formato yyyy-MM", k, F.col("mes_referencia").rlike("^[0-9]{4}-[0-9]{2}$"))

# Nulos (as variações são nulas só nos 12 primeiros meses)
for coluna in ["qtd_ucs_cadastradas", "qtd_ucs_faturadas", "consumo_total_kwh", "receita_energia", "receita_total", "preco_medio_kwh",
               "bandeira_predominante", "pct_faturas_bandeira_nao_verde", "qtd_faturas_em_aberto", "inadimplencia_pct",
               "valor_em_aberto_mais_90_dias", "qtd_ucs_com_gd", "qtd_interrupcoes", "dec_horas", "fec", "qtd_atendimentos", "nota_satisfacao_media"]:
    dq_check(t, f"{coluna} não nulo", k, F.col(coluna).isNotNull())
dq_check(t, "variações nulas de 2022-07 a 2023-06 e preenchidas depois", k,
         F.when(F.col("mes_referencia") <= P0_FIM,
                F.col("variacao_consumo_ano_anterior_pct").isNull() & F.col("variacao_receita_ano_anterior_pct").isNull())
          .otherwise(F.col("variacao_consumo_ano_anterior_pct").isNotNull() & F.col("variacao_receita_ano_anterior_pct").isNotNull()))

# Faixas
dq_check(t, "inadimplencia_pct entre 0 e 100", k, F.col("inadimplencia_pct").between(0, 100))
dq_check(t, "pct_faturas_bandeira_nao_verde entre 0 e 100", k, F.col("pct_faturas_bandeira_nao_verde").between(0, 100))
dq_check(t, "nota_satisfacao_media entre 1 e 5", k, F.col("nota_satisfacao_media").between(1, 5))
dq_check(t, "valores e contagens >= 0", k,
         (F.col("consumo_total_kwh") >= 0) & (F.col("receita_energia") >= 0) & (F.col("receita_total") >= F.col("receita_energia"))
         & (F.col("valor_em_aberto_mais_90_dias") >= 0) & (F.col("dec_horas") >= 0) & (F.col("fec") >= 0))
dq_check(t, "dec_horas e fec = 0 nos meses sem interrupção", k, (F.col("qtd_interrupcoes") > 0) | ((F.col("dec_horas") == 0) & (F.col("fec") == 0)))
dq_check(t, "faturadas <= cadastradas", k, F.col("qtd_ucs_faturadas") <= F.col("qtd_ucs_cadastradas"))

# Monotonia: a base e a GD só crescem
w_mes = Window.orderBy("mes_referencia")
mono = (k.withColumn("cad_ant", F.lag("qtd_ucs_cadastradas").over(w_mes)).withColumn("gd_ant", F.lag("qtd_ucs_com_gd").over(w_mes)))
dq_check(t, "qtd_ucs_cadastradas não diminui", mono, F.col("cad_ant").isNull() | (F.col("qtd_ucs_cadastradas") >= F.col("cad_ant")))
dq_check(t, "qtd_ucs_com_gd não diminui", mono, F.col("gd_ant").isNull() | (F.col("qtd_ucs_com_gd") >= F.col("gd_ant")))

# Reconciliação com a fato e com o diagnóstico
tot = k.agg(F.sum("consumo_total_kwh").alias("kwh"), F.sum("receita_energia").alias("rec"), F.sum("qtd_faturas_em_aberto").alias("abertas"),
            F.sum("qtd_atendimentos").alias("chamados"), F.sum("qtd_interrupcoes").alias("interr")).first()
fato_tot = fato.agg(F.sum("consumo_kwh").alias("kwh"), F.sum("valor_energia").alias("rec")).first()
dq_check_valor(t, "soma do consumo dos 48 meses = soma da fato", n_k, tot["kwh"], Decimal(fato_tot["kwh"]).quantize(Decimal("0.01")))
dq_check_valor(t, "soma da receita dos 48 meses = soma da fato", n_k, tot["rec"], fato_tot["rec"])

per = {r["p"]: r for r in k.withColumn("p", F.when(F.col("mes_referencia").between(P0_INI, P0_FIM), "p0")
                                                 .when(F.col("mes_referencia").between(P1_INI, P1_FIM), "p1"))
       .filter("p IS NOT NULL").groupBy("p").agg(F.sum("consumo_total_kwh").alias("kwh"), F.sum("receita_energia").alias("rec")).collect()}
dq_check_valor(t, "volume do primeiro ano = 227.052.927 kWh", n_k, per["p0"]["kwh"], Decimal("227052927.00"))
dq_check_valor(t, "volume dos últimos 12 meses = 218.079.054 kWh", n_k, per["p1"]["kwh"], Decimal("218079054.00"))
dq_check_valor(t, "receita do primeiro ano = R$ 117.830.394,38", n_k, per["p0"]["rec"], Decimal("117830394.38"))
dq_check_valor(t, "receita dos últimos 12 meses = R$ 127.262.595,65", n_k, per["p1"]["rec"], Decimal("127262595.65"))

dq_check_valor(t, "faturas em aberto (todos os meses) = 172.553", n_k, tot["abertas"], 172_553)
dq_check_valor(t, "chamados (todos os meses) = 318.871", n_k, tot["chamados"], 318_871)
dq_check_valor(t, "interrupções (todos os meses) = 180.000", n_k, tot["interr"], 180_000)
dq_check_valor(t, "interrupções em 2025-07 = 17.844 (evento climático)", n_k, k.filter("mes_referencia = '2025-07'").first()["qtd_interrupcoes"], 17_844)
dq_check_valor(t, "chamados em 2025-07 = 18.061", n_k, k.filter("mes_referencia = '2025-07'").first()["qtd_atendimentos"], 18_061)
ult = k.filter("mes_referencia = '2026-06'").first()
dq_check_valor(t, "UCs com GD em 2026-06 = 3.510", n_k, ult["qtd_ucs_com_gd"], 3_510)
dq_check_valor(t, "UCs cadastradas em 2026-06 = 45.000", n_k, ult["qtd_ucs_cadastradas"], 45_000)

# Informativo: horas interrompidas (UC x hora), para comparar com a fato_interrupcao (esperado: ~356.535 h)
print("horas interrompidas, soma de duracao_min / 60:", round(intr.agg(F.sum("duracao_min")).first()[0] / 60.0, 1))

# Bandeira no primeiro ano: 100% verde
dq_check(t, "primeiro ano: bandeira Verde e 0% de faturas não verdes", k,
         (F.col("mes_referencia") > P0_FIM) | ((F.col("bandeira_predominante") == "Verde") & (F.col("pct_faturas_bandeira_nao_verde") == 0)))
dq_check_valor(t, "20 colunas, como no contrato", n_k, len(df_kpis.columns), 20)

[PASS] gold.kpis_mensal | 48 meses | obtido 48 (esperado 48)
[PASS] gold.kpis_mensal | mes_referencia único | 0 chaves duplicadas de 48 linhas
[PASS] gold.kpis_mensal | primeiro mês = 2022-07 | obtido 2022-07 (esperado 2022-07)
[PASS] gold.kpis_mensal | último mês = 2026-06 | obtido 2026-06 (esperado 2026-06)
[PASS] gold.kpis_mensal | mes_referencia no formato yyyy-MM | 0/48 linhas falharam
[PASS] gold.kpis_mensal | qtd_ucs_cadastradas não nulo | 0/48 linhas falharam
[PASS] gold.kpis_mensal | qtd_ucs_faturadas não nulo | 0/48 linhas falharam
[PASS] gold.kpis_mensal | consumo_total_kwh não nulo | 0/48 linhas falharam
[PASS] gold.kpis_mensal | receita_energia não nulo | 0/48 linhas falharam
[PASS] gold.kpis_mensal | receita_total não nulo | 0/48 linhas falharam
[PASS] gold.kpis_mensal | preco_medio_kwh não nulo | 0/48 linhas falharam
[PASS] gold.kpis_mensal | bandeira_predominante não nulo | 0/48 linhas falharam
[PASS] gold.kpis_mensal | pct_faturas_bandeira_nao_verde não nulo | 0/48 lin

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[PASS] gold.kpis_mensal | qtd_ucs_cadastradas não diminui | 0/48 linhas falharam
[PASS] gold.kpis_mensal | qtd_ucs_com_gd não diminui | 0/48 linhas falharam
[PASS] gold.kpis_mensal | soma do consumo dos 48 meses = soma da fato | obtido 891449321.00 (esperado 891449321.00)
[PASS] gold.kpis_mensal | soma da receita dos 48 meses = soma da fato | obtido 499862082.73 (esperado 499862082.73)
[PASS] gold.kpis_mensal | volume do primeiro ano = 227.052.927 kWh | obtido 227052927.00 (esperado 227052927.00)
[PASS] gold.kpis_mensal | volume dos últimos 12 meses = 218.079.054 kWh | obtido 218079054.00 (esperado 218079054.00)
[PASS] gold.kpis_mensal | receita do primeiro ano = R$ 117.830.394,38 | obtido 117830394.38 (esperado 117830394.38)
[PASS] gold.kpis_mensal | receita dos últimos 12 meses = R$ 127.262.595,65 | obtido 127262595.65 (esperado 127262595.65)
[PASS] gold.kpis_mensal | faturas em aberto (todos os meses) = 172.553 | obtido 172553 (esperado 172553)
[PASS] gold.kpis_mensal | chamados (to

## 6. Registro das checagens e gravação, só se todas passarem

O resultado das checagens é gravado **sempre** em `silver.dq_log` (`table_name = gold.kpis_mensal`). A tabela só é gravada se **todas** passarem; se alguma falhar, o notebook interrompe com erro, a `_stg` fica para inspeção e a `kpis_mensal` anterior, se existir, permanece. Se todas passarem, a final é gravada a partir da `_stg`, e a `_stg` é apagada.

In [0]:
df_dq_log = spark.createDataFrame(dq_results)
df_dq_log.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(tabela_dq_log)

falhas = [r for r in dq_results if not r.passed]
if falhas:
    print(f"{len(falhas)} checagem(ns) falharam. {tabela_gold} NÃO foi gravada ({tabela_stg} ficou para inspeção):")
    for r in falhas:
        print(f"  - {r.check_name}: {n_br(r.failed_rows)} falha(s) em {n_br(r.total_rows)} linhas")
    raise Exception(f"{tabela_gold} não gravada: {len(falhas)} checagem(ns) de qualidade falharam (veja silver.dq_log)")

(
    k.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_gold)
)
spark.sql(f"DROP TABLE IF EXISTS {tabela_stg}")
print(f"Todas as {len(dq_results)} checagens passaram. {tabela_gold} gravada: {spark.table(tabela_gold).count()} linhas")

Todas as 46 checagens passaram. v_energy.gold.kpis_mensal gravada: 48 linhas


## 7. Os números do diretor, lidos da `kpis_mensal`

A pergunta do Henrique, respondida só com esta tabela: o consumo e a receita dos **últimos 12 meses** contra o **primeiro ano**, mais o preço, a inadimplência e o atendimento. Esperado: volume **−3,95%**, receita de energia **+8,00%** e preço médio **+12,45%**.

Os cards do dashboard devem calcular o preço dos períodos como `receita ÷ volume` (somando antes de dividir), e não como a média dos preços mensais.

In [0]:
kp = spark.table(tabela_gold)
p = (kp.withColumn("periodo", F.when(F.col("mes_referencia").between(P0_INI, P0_FIM), "1. primeiro ano")
                               .when(F.col("mes_referencia").between(P1_INI, P1_FIM), "2. últimos 12 meses"))
     .filter("periodo IS NOT NULL").groupBy("periodo")
     .agg(F.sum("consumo_total_kwh").alias("volume_kwh"), F.sum("receita_energia").alias("receita_energia"),
          F.sum("receita_total").alias("receita_total"), F.sum("qtd_faturas_em_aberto").alias("faturas_em_aberto"),
          F.sum("qtd_atendimentos").alias("chamados"), F.sum("qtd_interrupcoes").alias("interrupcoes"),
          F.round(F.avg("inadimplencia_pct"), 2).alias("inadimplencia_media_pct"),
          F.round(F.avg("nota_satisfacao_media"), 2).alias("nota_media_dos_meses"))
     .withColumn("preco_medio", F.round(F.col("receita_energia") / F.col("volume_kwh"), 4)).orderBy("periodo"))
display(p)
# esperado: volume 227.052.927 -> 218.079.054 (-3,95%); receita de energia 117,83 mi -> 127,26 mi (+8,00%); preço 0,5190 -> 0,5836 (+12,45%)
# chamados 76.561 -> 88.823 (+16,0%); faturas em aberto, ~8% das faturas em cada período

# O mês de julho de 2025: chamados e interrupções fora do padrão
display(kp.filter("mes_referencia BETWEEN '2025-05' AND '2025-09'")
        .select("mes_referencia", "qtd_atendimentos", "nota_satisfacao_media", "qtd_interrupcoes", "dec_horas", "fec"))
# esperado: 2025-07 com 18.061 chamados e nota ~3,11; interrupções ~5,3x o normal

# Bandeira e GD ao longo do tempo
display(kp.select("mes_referencia", "bandeira_predominante", "pct_faturas_bandeira_nao_verde", "qtd_ucs_com_gd", "inadimplencia_pct",
                  "valor_em_aberto_mais_90_dias", "variacao_consumo_ano_anterior_pct", "variacao_receita_ano_anterior_pct").orderBy("mes_referencia"))

periodo,volume_kwh,receita_energia,receita_total,faturas_em_aberto,chamados,interrupcoes,inadimplencia_media_pct,nota_media_dos_meses,preco_medio
1. primeiro ano,227052927.00,117830394.38,157107191.43,42976,76561,40172,5.48,4.10,0.5190
2. últimos 12 meses,218079054.00,127262595.65,169683458.59,43294,88823,58777,5.56,4.02,0.5836


mes_referencia,qtd_atendimentos,nota_satisfacao_media,qtd_interrupcoes,dec_horas,fec
2025-08,6504,4.10,3409,0.1280,0.0758
2025-09,6477,4.10,3289,0.1168,0.0731
2025-07,18061,3.11,17844,1.9440,0.3970
2025-06,6334,4.10,3354,0.1271,0.0746
2025-05,6319,4.10,3411,0.1263,0.0759


mes_referencia,bandeira_predominante,pct_faturas_bandeira_nao_verde,qtd_ucs_com_gd,inadimplencia_pct,valor_em_aberto_mais_90_dias,variacao_consumo_ano_anterior_pct,variacao_receita_ano_anterior_pct
2022-07,Verde,0.00,0,5.35,509464.28,null,null
2022-08,Verde,0.00,0,5.15,492512.64,null,null
2022-09,Verde,0.00,0,5.23,551533.82,null,null
2022-10,Verde,0.00,0,5.69,715635.45,null,null
2022-11,Verde,0.00,0,5.43,777749.98,null,null
2022-12,Verde,0.00,0,5.47,820674.15,null,null
2023-01,Verde,0.00,0,5.68,888126.89,null,null
2023-02,Verde,0.00,0,5.86,887367.40,null,null
2023-03,Verde,0.00,0,5.74,877944.04,null,null
2023-04,Verde,0.00,0,5.97,856671.54,null,null
